<a href="https://colab.research.google.com/github/PedroK0/Intelig-ncia-Artificial-e-Programa-o-Para-Engenharia/blob/main/exercicio_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Exercício 20: Sistema de Intertravamento e Deslastre em Lista de Cargas Prioritárias

Nesta seção, modelamos um sistema SCADA de despacho e deslastre de cargas. Quando a soma das correntes ativas excede a capacidade do transformador ($150\text{ A}$), o algoritmo analisa as prioridades (onde prioridade `4` é a menos essencial e `1` é a mais crítica/vital) e realiza o desligamento ordenado das cargas estritamente necessárias até restabelecer a segurança operativa.
```

In [6]:
# 1. Definição do banco de dados de cargas do barramento principal
cargas = [
    {"circuito": "UTI_HOSPITALAR", "corrente_a": 60.0, "prioridade": 1, "ligado": True},
    {"circuito": "ILUMINACAO_PUBLICA", "corrente_a": 45.0, "prioridade": 3, "ligado": True},
    {"circuito": "BOMBEAMENTO_ESGOTO", "corrente_a": 55.0, "prioridade": 2, "ligado": True},
    {"circuito": "CLUBE_RECREATIVO", "corrente_a": 35.0, "prioridade": 4, "ligado": True}
]

LIMITE_MAXIMO_A = 150.0

# Função para calcular a corrente total ativa no sistema
def calcular_corrente_total(lista_cargas):
    return sum(carga["corrente_a"] for carga in lista_cargas if carga["ligado"])

corrente_inicial = calcular_corrente_total(cargas)
print(f"--- ESTADO INICIAL DO BARRAMENTO ---")
print(f"Corrente Total Ativa: {corrente_inicial:.1f} A (Limite: {LIMITE_MAXIMO_A:.1f} A)\n")

# 2. Algoritmo de Deslastre de Emergência se o limite for violado
if corrente_inicial > LIMITE_MAXIMO_A:
    print("[ALERTA] Sobrecarga detectada! Iniciando protocolo de deslastre automático...\n")

    # Ordenamos as cargas de menor prioridade para a maior (número de prioridade decrescente, ex: 4, 3, 2, 1)
    cargas_ordenadas_para_corte = sorted(cargas, key=lambda x: x["prioridade"], reverse=True)

    for carga in cargas_ordenadas_para_corte:
        corrente_atual = calcular_corrente_total(cargas)
        if corrente_atual <= LIMITE_MAXIMO_A:
            break  # Barramento estabilizado na faixa de segurança

        if carga["ligado"]:
            carga["ligado"] = False
            corrente_apos_corte = calcular_corrente_total(cargas)
            print(f"-> DESLIGADO: Circuito '{carga['circuito']}' (Corte de {carga['corrente_a']:.1f} A)")
            print(f"   Nova corrente de barramento: {corrente_apos_corte:.1f} A\n")

# 3. Relatório Final de Operação Pós-Incidente
print("--- ESTADO FINAL APÓS DESLASTRE ---")
corrente_final = calcular_corrente_total(cargas)
print(f"Corrente Final Operativa: {corrente_final:.1f} A (Status: SEGURA)\n")

for c in cargas:
    status_txt = "LIGADO" if c["ligado"] else "DESLIGADO (DESLASTREADO)"
    print(f"Circuito: {c['circuito']:<20} | Prioridade: {c['prioridade']} | Status: {status_txt}")

--- ESTADO INICIAL DO BARRAMENTO ---
Corrente Total Ativa: 195.0 A (Limite: 150.0 A)

[ALERTA] Sobrecarga detectada! Iniciando protocolo de deslastre automático...

-> DESLIGADO: Circuito 'CLUBE_RECREATIVO' (Corte de 35.0 A)
   Nova corrente de barramento: 160.0 A

-> DESLIGADO: Circuito 'ILUMINACAO_PUBLICA' (Corte de 45.0 A)
   Nova corrente de barramento: 115.0 A

--- ESTADO FINAL APÓS DESLASTRE ---
Corrente Final Operativa: 115.0 A (Status: SEGURA)

Circuito: UTI_HOSPITALAR       | Prioridade: 1 | Status: LIGADO
Circuito: ILUMINACAO_PUBLICA   | Prioridade: 3 | Status: DESLIGADO (DESLASTREADO)
Circuito: BOMBEAMENTO_ESGOTO   | Prioridade: 2 | Status: LIGADO
Circuito: CLUBE_RECREATIVO     | Prioridade: 4 | Status: DESLIGADO (DESLASTREADO)


```markdown
## Explicação do Código para a Apresentação

* **`calcular_corrente_total()`**: Função utilitária que usa uma expressão geradora rápida para somar a corrente apenas dos circuitos que estão ativos (`"ligado": True`).
* **`sorted(cargas, key=lambda x: x["prioridade"], reverse=True)`**:
  * Esta é a chave lógica do algoritmo. A função ordena a nossa lista de cargas com base no campo `prioridade`.
  * Como o parâmetro `reverse=True` está ativo, a lista é organizada do maior número (`4` - menos prioritário/clube recreativo) ao menor número (`1` - mais prioritário/hospital).
* **Laço de Deslastre (`for`)**:
  * O algoritmo avalia sequencialmente as cargas ordenadas de menor importância.
  * Se a corrente total atual ainda exceder o limite de segurança, a carga é desativada (`"ligado" = False`) e o novo estado de corrente do barramento é imediatamente calculado e exibido.
  * O laço é interrompido (`break`) assim que a corrente total atinge um valor seguro ($\le 150,0\text{ A}$), poupando os demais circuitos essenciais.
```